In [ ]:
import pandas as pd

X_train_original = pd.read_csv('/content/X_train_original_no_scaling (1).csv')
display(X_train_original.head())
display(X_train_original.info())

In [ ]:
import numpy as np
class my_NN(object):
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size, lr, lambd):
        self.input = input_size
        self.hidden1_units = hidden1_size
        self.hidden2_units = hidden2_size
        self.output = output_size
        self.lr = lr
        self.lambd = lambd

        # initialize matrix of weights;
        np.random.seed(1)
        # weight1: input -> hidden layer 1
        self.w1 = np.random.randn(self.input, self.hidden1_units)
        # weight2: hidden layer 1 -> hidden layer 2
        self.w2 = np.random.randn(self.hidden1_units, self.hidden2_units)
        # weight3: hidden layer 2 -> output
        self.w3 = np.random.randn(self.hidden2_units, self.output)

    def _forward_propagation(self, X):
        self.z2 = np.dot(self.w1.T, X.T)
        self.a2 = self._tanh(self.z2)
        self.z3 = np.dot(self.w2.T, self.a2)
        self.a3 = self._tanh(self.z3)
        self.z4 = np.dot(self.w3.T, self.a3)
        self.a4 = self._tanh(self.z4)
        return self.a4

    def _tanh(self, z):
        return (np.exp(z)-np.exp(-z))/(np.exp(z)+np.exp(-z))

    def _tanh_prime(self, z):
        return 1 - np.tanh(z)**2

    def _loss(self, predict, y):
        m = y.shape[0]
        y = y.reshape(1, -1)
        sq_error = np.multiply((predict - y), (predict - y))
        mse = np.sum(sq_error) / m
        reg = (self.lambd / (2 * m)) * (np.sum(self.w1 ** 2) + np.sum(self.w2 ** 2) + np.sum(self.w3 ** 2))
        loss = mse + reg
        return loss

    def _backward_propagation(self, X, y):
        predict = self._forward_propagation(X)
        m = X.shape[0]
        y = y.reshape(1, -1)

        delta4 = np.multiply((predict - y), self._tanh_prime(self.z4))
        self.dw3 = (1/m) * np.dot(self.a3, delta4.T) + (self.lambd/m) * self.w3
        delta3 = np.multiply(np.dot(self.w3, delta4), self._tanh_prime(self.z3))
        self.dw2 = (1/m) * np.dot(self.a2, delta3.T) + (self.lambd/m) * self.w2
        delta2 = np.multiply(np.dot(self.w2, delta3), self._tanh_prime(self.z2))
        self.dw1 = (1/m) * np.dot(X.T, delta2.T) + (self.lambd/m) * self.w1


    def train(self, X, y, iteration=33):
        for i in range(iteration):
            y_hat = self._forward_propagation(X)
            loss = self._loss(y_hat, y)
            self._backward_propagation(X,y)
            self._update_parameters()
            if i%10==0:
                print("loss: ", loss)

    def predict(self, X):
        y_hat = self._forward_propagation(X)
        y_hat = [1 if i[0] >= 0.5 else 0 for i in y_hat.T]
        return np.array(y_hat)

    def score(self, predict, y):
        cnt = np.sum(predict==y)
        return (cnt/len(y))*100

    def _update_parameters(self):
        self.w1 = self.w1 - self.lr * self.dw1
        self.w2 = self.w2 - self.lr * self.dw2
        self.w3 = self.w3 - self.lr * self.dw3

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('/content/X_train_original_no_scaling (1).csv')

# features and target
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# convert to float
X = X.astype(float)
y = y.astype(float)

# scale input features
x_scaler = StandardScaler()
X = x_scaler.fit_transform(X)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (56902, 21)
y shape: (56902,)


In [ ]:
nn = my_NN(
    input_size=X.shape[1],
    hidden1_size=16,
    hidden2_size=8,
    output_size=1,
    lr=0.01,
    lambd=0.001
)

epochs = 2000

for epoch in range(epochs):
    pred = nn._forward_propagation(X)
    loss = nn._loss(pred, y)

    nn._backward_propagation(X, y)
    nn._update_parameters()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss = {loss}")

Epoch 0, Loss = 1988356.5284714806
Epoch 100, Loss = 1985734.414227891
Epoch 200, Loss = 1985733.0098465807
Epoch 300, Loss = 1985732.7930350015
Epoch 400, Loss = 1985732.696312242
Epoch 500, Loss = 1985732.6256142866
Epoch 600, Loss = 1985732.5507357547
Epoch 700, Loss = 1985732.4835344108
Epoch 800, Loss = 1985732.4459268511
Epoch 900, Loss = 1985732.4264399728
Epoch 1000, Loss = 1985732.4120118131
Epoch 1100, Loss = 1985732.4005499864
Epoch 1200, Loss = 1985732.3911296919
Epoch 1300, Loss = 1985732.3832147028
Epoch 1400, Loss = 1985732.3764559252
Epoch 1500, Loss = 1985732.3706101498
Epoch 1600, Loss = 1985732.3654997125
Epoch 1700, Loss = 1985732.3609902046
Epoch 1800, Loss = 1985732.3569771117
Epoch 1900, Loss = 1985732.3533765282


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import seaborn as sns

# Perform PCA to reduce to 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Create a DataFrame for plotting
pca_df = pd.DataFrame(data = X_pca, columns = ['principal component 1', 'principal component 2'])
pca_df['target'] = y

plt.figure(figsize=(10,8))
sns.scatterplot(
    x='principal component 1',
    y='principal component 2',
    hue='target',
    palette=sns.color_palette("viridis", as_cmap=True),
    data=pca_df,
    legend='full',
    alpha=0.7
)
plt.title('2 Component PCA of X_train_original Data (colored by target)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid()
plt.show()


In [ ]:
import pandas as pd

test_df = pd.read_csv('/content/X_test_original_no_scaling (1).csv')   # your test file

X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

# Scale X_test using the same scaler fitted on X_train
X_test = x_scaler.transform(X_test)

pred_test = nn._forward_propagation(X_test)
pred_test = pred_test.reshape(-1)

# save predictions
import pandas as pd

output = pd.DataFrame({"Prediction": pred_test})
output.to_csv('/content/predictions.csv', index=False)

print(output.head())

   Prediction
0    0.999999
1    0.999981
2    1.000000
3    0.999991
4    1.000000
